--- DATA QUALITY REPORT ---

This dataset contains 1,067,371 transaction line items across 9 columns, spanning December 1, 2009 to December 9, 2011, covering 5,942 unique customers, 5,305 unique products, and 43 countries. Data quality issues identified: 243,007 rows (~23%) missing Customer ID, 4,382 missing Description, 12,133 exact duplicate rows, 22,950 rows with negative Quantity (returns/adjustments, only partially aligned with the 19,494 invoices formally marked as cancelled via a 'C' prefix), and 5 rows with negative Price. Additionally, the StockCode field contains non-product entries (POST, DOT, M, D, S, ADJUST, AMAZONFEE, CRUK, etc.) representing postage, fees, discounts, and manual adjustments rather than actual gift-ware sales — these will be excluded from revenue and AOV calculations. InvoiceDate is currently stored as text and will need conversion to datetime for time-based analysis.

In [1]:
import pandas as pd

In [2]:
df_2009_2010 = pd.read_excel(r"C:\Users\DELL\Desktop\d2c-revenue-growth-simulator\data\raw\online_retail_II.xlsx",
                             sheet_name='Year 2009-2010',engine='calamine')

df_2010_2011 = pd.read_excel(r"C:\Users\DELL\Desktop\d2c-revenue-growth-simulator\data\raw\online_retail_II.xlsx",
                             sheet_name='Year 2010-2011',engine='calamine')

print ("done loading")

done loading


In [3]:
df_2009_2010.to_csv(r"C:\Users\DELL\Desktop\d2c-revenue-growth-simulator\data\raw\online_retail_2009_2010.csv", index=False)
df_2010_2011.to_csv(r"C:\Users\DELL\Desktop\d2c-revenue-growth-simulator\data\raw\online_retail_2010_2011.csv", index=False)
print("Saved to CSV")

Saved to CSV


In [4]:
df_2009_2010 = pd.read_csv(r"C:\Users\DELL\Desktop\d2c-revenue-growth-simulator\data\raw\online_retail_2009_2010.csv")
df_2010_2011 = pd.read_csv(r"C:\Users\DELL\Desktop\d2c-revenue-growth-simulator\data\raw\online_retail_2010_2011.csv")
print(df_2009_2010.shape)

(525461, 8)


In [5]:
df_2009_2010['SourcePeriod'] = '2009-2010'
df_2010_2011['SourcePeriod'] = '2010-2011'
df_raw = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
print(df_raw.shape)

(1067371, 9)


In [6]:
df_raw.info()
df_raw.head(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   Invoice       1067371 non-null  object 
 1   StockCode     1067371 non-null  object 
 2   Description   1062989 non-null  object 
 3   Quantity      1067371 non-null  int64  
 4   InvoiceDate   1067371 non-null  object 
 5   Price         1067371 non-null  float64
 6   Customer ID   824364 non-null   float64
 7   Country       1067371 non-null  object 
 8   SourcePeriod  1067371 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 73.3+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2009-2010
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2009-2010
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,2009-2010
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,2009-2010


In [7]:
df_raw.isnull().sum()

Invoice              0
StockCode            0
Description       4382
Quantity             0
InvoiceDate          0
Price                0
Customer ID     243007
Country              0
SourcePeriod         0
dtype: int64

In [8]:
(df_raw['Quantity'] < 0).sum()


np.int64(22950)

In [9]:
(df_raw['Price'] < 0).sum()

np.int64(5)

In [10]:
df_raw['Invoice'].astype(str).str.startswith('C').sum()

np.int64(19494)

In [11]:
df_raw.duplicated().sum()

np.int64(12133)

In [12]:
print("Customers:", df_raw['Customer ID'].nunique())
print("Products:", df_raw['StockCode'].nunique())
print("Countries:", df_raw['Country'].nunique())

Customers: 5942
Products: 5305
Countries: 43


In [13]:
df_raw['InvoiceDate'].min(), df_raw['InvoiceDate'].max()

('2009-12-01 07:45:00', '2011-12-09 12:50:00')

In [14]:
df_raw['StockCode'].value_counts().head(20)

StockCode
85123A    5829
22423     4424
85099B    4216
21212     3318
20725     3259
84879     2960
47566     2768
21232     2747
22197     2549
22383     2540
20727     2529
21931     2434
22386     2347
22469     2325
22411     2297
84991     2271
22382     2251
22384     2230
21080     2224
22086     2217
Name: count, dtype: int64

In [15]:
df_raw[df_raw['StockCode'].str.match(r'^[A-Za-z]+$', na=False)]['StockCode'].value_counts()

StockCode
POST         2122
DOT          1446
M            1421
D             177
S             104
ADJUST         67
AMAZONFEE      43
DCGSSGIRL      25
DCGSSBOY       23
PADS           19
CRUK           16
B               6
m               5
GIFT            1
DCGSLBOY        1
DCGSLGIRL       1
Name: count, dtype: int64

---data cleaning---

In [16]:
df = df_raw.copy()
print(df.shape)

(1067371, 9)


In [17]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)

datetime64[ns]


In [18]:
df[df['Price']<0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,2009-2010
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,2010-2011


In [20]:
df = df[df['Price'] >= 0]
print(df.shape)

(1067366, 9)


In [21]:
df[df['Price'] < 0].shape

(0, 9)

In [22]:
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(df.shape)

(1047872, 9)


In [23]:
df = df[df['Quantity'] > 0]
print(df.shape)

(1044415, 9)


In [24]:
non_product_codes = ['POST', 'DOT', 'M', 'm', 'D', 'S', 'ADJUST', 'AMAZONFEE',
                      'CRUK', 'B', 'GIFT', 'DCGSSGIRL', 'DCGSSBOY',
                      'DCGSLBOY', 'DCGSLGIRL', 'PADS']
df = df[~df['StockCode'].isin(non_product_codes)]
print(df.shape)

(1040078, 9)


In [25]:
df = df[df['Customer ID'].notnull()]
print(df.shape)

(802995, 9)


In [26]:
df = df.drop_duplicates()
print(df.shape)

(791065, 9)


In [27]:
df['Customer ID'] = df['Customer ID'].astype(int)   # was float due to NaNs, now safe to convert
df['Invoice'] = df['Invoice'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 791065 entries, 0 to 1067369
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Invoice       791065 non-null  object        
 1   StockCode     791065 non-null  object        
 2   Description   791065 non-null  object        
 3   Quantity      791065 non-null  int64         
 4   InvoiceDate   791065 non-null  datetime64[ns]
 5   Price         791065 non-null  float64       
 6   Customer ID   791065 non-null  int64         
 7   Country       791065 non-null  object        
 8   SourcePeriod  791065 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(5)
memory usage: 60.4+ MB


In [28]:
df['Revenue'] = df['Quantity'] * df['Price']

In [29]:
print("Final shape:", df.shape)
print("Remaining nulls:\n", df.isnull().sum())
print("Date range:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())
print("Unique customers:", df['Customer ID'].nunique())
print("Total revenue:", df['Revenue'].sum())

Final shape: (791065, 10)
Remaining nulls:
 Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price           0
Customer ID     0
Country         0
SourcePeriod    0
Revenue         0
dtype: int64
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5856
Total revenue: 17391102.891000003


In [30]:
df.to_csv('../data/processed/online_retail_cleaned.csv', index=False)
print("Saved cleaned dataset")

Saved cleaned dataset
